In [ ]:
# fix relative imports
import os
cwd = os.path.normpath(os.getcwd())
cwd = cwd.split(os.sep)
find = cwd.index("fidelity-phase-tran")
newdir = f"{os.sep}".join(cwd[:find+1])
os.chdir(newdir)
%load_ext autoreload
%autoreload 2

# import known packages
import numpy as np
import pickle
import gzip

from matplotlib import pyplot as plt

from scipy import signal
from scipy.optimize import curve_fit

# import adhoc packages
from qphaset.phases import gstates_to_rdms_matrix_qs_mps, phases_vfield
from qphaset.plotting import plot_grad_g_angle_stream

# Order Parameter Analysis

In [ ]:
"""
Type of filename to use:
filename = 'results/data/{model}_{identifier}.pkl'
"""
# filename = 'results/annni-20spins-64x64-dmrg-20240212.pkl'             # First decent ANNNI 20
# filename = 'results/ising-20spins-64x64-dmrg-20240214.pkl'

# filename = "annni_ext-93346fe8-cf92-4c55-9b47-0cb2b6f25c0c.pkl"        # ANNNI 50, c1=-1, upside down (*)
# filename = "annni_ext-7e477513-fe51-4ee3-9ae3-75766993ae7a.pkl"        # ANNNI 50, c1=h-1, upside down
# filename = "annni_ext-5234439f-02a3-4e3a-8c6a-a5d0a670dd0c.pkl"        # ANNNI 50, c1=-1, upside down, floating detail
# filename = "annni_ext-d02bd810-dece-40b1-8e36-ac6c8164bb46.pkl"        # ANNNI 50, c1=-0.1, upside down, floating detail
# filename = "annni_ext-aba4d4c0-cb6f-454b-b2e4-8babb3be68b6.pkl"        # ANNNI 50, c1=0, upside down, floating detail, 16 x 16
# filename = "annni_ext-0b44047c-884a-42b9-8bfe-bc5beb29be96.pkl"        # ANNNI 30, c1=0, upside down, floating detail, 32 x 32

# Effects of the grid coarse on the floating phase.
# filename = "annni_ext-b00ffcec-f325-4f18-b5d8-e15111ee72fb.pkl"            # ANNNI 30, c1=-0.1, upside down, floating detail, 32 x 32 (*)
# filename = "annni_ext-a76f14e2-354c-4588-a698-5d7cdaef11ec.pkl"            # ANNNI 30, c1=-0.1, upside down, floating detail, 64 x 64 (*)
# filename = "annni_ext-c47e7c80-227f-4b18-b8a4-c910cc6d6908.pkl"            # ANNNI 50, c1=-1, upside down, floating detail, 64 x 64 (**)
filename = "annni_ext-b9e5a93e-f105-411d-8013-b3120de2b88e.pkl"            # ANNNI 50, c1=-1, upside down, floating detail + multi-critical point, 64 x 64 (**)

# filename = "annni_ext-516fe606-16d0-4bce-afd0-5af6fb262d91.pkl"              # ANNNI 50, c1=-1, upside down, floating detail ++ (**)
# filename = "annni_ext-fad679a7-c06a-4112-8f28-97c92414d43b.pkl"                # ANNNI 30, c1=-1, upside down, floating detail ++ (**)

# filename = "annni_ext-297aa19b-e072-4ec5-b9fc-718bf75db45e.pkl"             # ANNNI 16, c1=-1, upside down, 64 x 64
# filename = "annni_ext-e810f2eb-6837-48a9-a2c7-7acd305f5e70.pkl"             # ANNNI 20, c1=-1, upside down, 64 x 64 (*)

# francesco's qs-mps results
filename = "results/data/ANNNI-7902ff06-aac2-4fab-ac2a-2692af269fd4.pkl" # ANNNI 12, c1=-1, upside down, 64 x 64
# filename = "results/data/ANNNI-8fad8665-b3af-43a1-be1b-c27fa8b42920.pkl"
filename = "results/data/ANNNI-8e2b8bce-1d27-41d0-a48b-fbce8d4ab8cc.pkl" # ANNNI 50, c1=-1, upside down, 64 x 64
# filename = "results/data/ANNNI-010201b6-5ab8-45af-9e50-507c9b609272.pkl" # if we truncate the bond dimension dmrg finds a product state, we can only see the disorder line

with gzip.open(filename, 'rb') as f:
    data = pickle.load(f)
params = data['params']
l, n = data['l'], data['n']
gstates = data['gstates']
stats = data['stats']

params_extent = np.concatenate([np.min(params, axis=0), np.max(params, axis=0)])
params_extent = tuple(params_extent[[0, 2, 1, 3]])

## Order Parameter in the Ising Phase

## Order Parameter in the Floating Phase

A first attempt could be to look at the expectation value of the order parameter with respect to rdms.  
By fixing the tuple of parameters $(\kappa , h)$, we take 2qubit-rdms for different sites in the chain (avoiding the borders).  
We would like to see Friedel oscillation and extract the Luttinger liquid parameter $K$ with which we can detect the incommensurate-incommensurate BKT transition.

In [ ]:
extent_sites = [9,40]
params_h_05 = params.reshape((64,64,2))[32]
gstates_1d = np.asarray(gstates).reshape((64,64))[32]
# every element of rdms_h_05 is a list of rdms of all the kappas in the horizontal line h=0.5 
rdms_h_05 = []
for j in range(extent_sites[0],extent_sites[1]):
    sites = [j,j+1]
    rdms_j = gstates_to_rdms_matrix_qs_mps(gstates_1d, sites=sites, shape=(gstates_1d.shape))
    print(rdms_j.shape)
    rdms_h_05.append(rdms_j)

rdms_h_05 = np.array(rdms_h_05)

In [ ]:
params_h_05 = params.reshape((64,64,2))[45]
params_h_05[63] # this point of the phase diagram should belong to the paramagnetic phase

In [ ]:
rdm_example = rdms_h_05[:,63]

### Different observables

In [ ]:
obs_discovery = np.array([
        [-0.15633863, -0.12925051, -0.10687629,  0.2619572 ],
        [-0.12925051,  0.23224414,  0.35059704, -0.27080356],
        [-0.10687629,  0.35059704,  0.23549051, -0.29999286],
        [ 0.2619572 , -0.27080356, -0.29999286,  0.31649385]
                        ])

In [ ]:
obs_1 = np.array([[ 0.02140907, -0.1003536 ,  0.10424755, -0.00351075],
       [-0.1003536 ,  0.47040086, -0.48865349,  0.0164564 ],
       [ 0.10424755, -0.48865349,  0.50761436, -0.01709494],
       [-0.00351075,  0.0164564 , -0.01709494,  0.00057571]])

In [ ]:
obs = np.array([[0.10720876, 0.15911587, 0.13877098, 0.2261411 ],
       [0.15911587, 0.23615477, 0.20595952, 0.33563151],
       [0.13877098, 0.20595952, 0.1796251 , 0.29271696],
       [0.2261411 , 0.33563151, 0.29271696, 0.47701137]]) # take the observable from the OPD notebook

idx = 63
rdm_example = rdms_h_05[:,idx] # we take the first list of rdms relative to 29 different sites and k=0.64 = 0.50
exp_vals = [np.trace(obs @ rdm).real for rdm in rdm_example]

plt.plot(range(extent_sites[0],extent_sites[1]), exp_vals, '--')
plt.title("Tr($O \\rho$)"+f"  $k={round(params_h_05[idx][0],2)}, h={round(params_h_05[idx][1],2)}$")
plt.xlabel("$j$ sites")

In [ ]:
def cosfit(x,w,phi,A,c):
    return A*np.cos(w*x+phi) + c

popt,copt = curve_fit(cosfit, np.linspace(extent_sites[0],extent_sites[1],47), exp_vals, p0=[1*np.pi,np.pi,0.001,0.2045])
popt

In [ ]:
xs = np.linspace(extent_sites[0],extent_sites[1],500)
idx = 63
rdm_example = rdms_h_05[:,idx] # we take the first list of rdms relative to 29 different sites and k=0.64 = 0.50
exp_vals = [np.trace(obs*rdm).real for rdm in rdm_example]
plt.plot(range(extent_sites[0],extent_sites[1]), exp_vals, '--', label="data")
plt.plot(xs, cosfit(xs,1*np.pi,np.pi,1e-3,0.2045))

### Fourier Analysis of the oscillations

In [ ]:
idx = 63
rdm_example = rdms_h_05[:,idx] # we take the first list of rdms relative to 29 different sites and k=0.64 = 0.50
exp_vals = [np.trace(obs @ rdm).real for rdm in rdm_example]
signal = exp_vals - np.mean(exp_vals)

# Compute the Fourier Transform
fourier_transform = np.fft.fft(signal)

# Compute the frequencies corresponding to the Fourier Transform
frequencies = np.fft.fftfreq(len(signal))

# Compute the magnitude and phase of the Fourier Transform
magnitude = np.abs(fourier_transform)
phase = np.angle(fourier_transform)

# Plot the original signal
plt.figure(figsize=(12, 8))

plt.subplot(3, 1, 1)
plt.stem(signal)
plt.title('Original Signal')
plt.xlabel('Sample')
plt.ylabel('Amplitude')

# Plot the magnitude spectrum
plt.subplot(3, 1, 2)
plt.stem(frequencies, magnitude)
plt.title('Magnitude Spectrum')
plt.xlabel('Frequency')
plt.ylabel('Magnitude')

# Plot the phase spectrum
plt.subplot(3, 1, 3)
plt.stem(frequencies, phase)
plt.title('Phase Spectrum')
plt.xlabel('Frequency')
plt.ylabel('Phase (radians)')

plt.tight_layout()
plt.show()

In [ ]:
np.max(phase), 0.8*np.pi

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].set_title("magnitude")
ax[0].plot(frequencies, magnitude)
ax[0].set_yscale('log')

ax[1].set_title("phase")
ax[1].plot(frequencies, phase)